<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [10]</a>'.</span>

In [1]:
import pandas as pd
import numpy as np

In [2]:
country_vehicle = pd.read_excel('./Data Extraction Sheet.xlsx', sheet_name="Country-Vehicle Extraction")
country_vehicle = country_vehicle[country_vehicle.Country == 'Nigeria']
country_vehicle

/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/0100_data_prep/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,Country,Vehicle,Quintile,Data need,Data point name,Units,Year,Data source,Value,CI,SE,Notes
42,Nigeria,Bouillon,All,Vehicle consumption by WRA -- any,percentage,%,2021.0,NFCMS,0.981000,"[97.5,98.8]",NaN,https://www.unicef.org/nigeria/media/9271/file...
43,Nigeria,Bouillon,Lowest,Vehicle consumption by WRA -- any,percentage,%,2021.0,NFCMS,0.963000,"[94.2, 98.4]",NaN,"Table 169, page 220"
44,Nigeria,Bouillon,Second,Vehicle consumption by WRA -- any,percentage,%,2021.0,NFCMS,0.981000,"[96.8, 99.3]",NaN,95% CI
45,Nigeria,Bouillon,Middle,Vehicle consumption by WRA -- any,percentage,%,2021.0,NFCMS,0.992000,"[98.5, 99.8]",NaN,"These numbers are for non-pregnant WRA, not al..."
46,Nigeria,Bouillon,Fourth,Vehicle consumption by WRA -- any,percentage,%,2021.0,NFCMS,0.986000,"[97.8, 99.4]",NaN,NaN
47,Nigeria,Bouillon,Highest,Vehicle consumption by WRA -- any,percentage,%,2021.0,NFCMS,0.986000,"[97.5, 99.7]",NaN,NaN
48,Nigeria,Bouillon,All,Vehicle consumption by WRA -- amount,mean,g/day,2021.0,NFCMS,6.300000,"[6.0, 6.6]",0.1,"Table 170, page 222"
49,Nigeria,Bouillon,Lowest,Vehicle consumption by WRA -- amount,mean,g/day,2021.0,NFCMS,8.400000,"[7.7, 9.0]",0.3,95% CI
50,Nigeria,Bouillon,Second,Vehicle consumption by WRA -- amount,mean,g/day,2021.0,NFCMS,8.000000,"[7.3, 8.7]",0.3,Pregnant and non-pregnant WRA have been strati...
51,Nigeria,Bouillon,Middle,Vehicle consumption by WRA -- amount,mean,g/day,2021.0,NFCMS,5.900000,"[5.3, 6.5]",0.3,"Also, these numbers are across all women, not ..."


In [3]:
assert (country_vehicle.Vehicle == 'Bouillon').all()

In [4]:
country_vehicle['Data need'].unique()

array(['Vehicle consumption by WRA -- any',
       'Vehicle consumption by WRA -- amount',
       'Vehicle consumption by U5 children -- any',
       'Vehicle consumption by U5 children -- amount',
       'Vehicle "fortifiability" (essentially amount industrially produced)'],
      dtype=object)

In [5]:
percent_data_needs = {
    "Vehicle consumption by WRA -- any": "consumed_any_vehicle",
    'Vehicle "fortifiability" (essentially amount industrially produced)': "vehicle_fortifiability",
}

In [6]:
pregnancy_sim_data_dir = '../../0200_pregnancy_sim/src/vivarium_gates_lsff_by_wealth_quintile/data/raw_data/'
import pathlib
pathlib.Path(pregnancy_sim_data_dir).mkdir(parents=True, exist_ok=True)

In [7]:
def check_with_total(rows):
    assert rows.Quintile.is_unique
    rows = rows.set_index('Quintile').Value
    total = rows.loc['All']
    print(f'Total: {total}')
    by_quintile = rows[rows.index != 'All']
    print('By quintile')
    print(by_quintile)
    assert np.isclose(by_quintile.mean(), total, atol=0, rtol=0.1)
    print(f'Implied total with equal weight: {by_quintile.mean()}')
    by_quintile = by_quintile.rename("value").reset_index().rename(columns={"Quintile": "wealth_quintile"})
    by_quintile["wealth_quintile"] = by_quintile.wealth_quintile.str.lower()
    return by_quintile

In [8]:
for data_need_name, dir_name in percent_data_needs.items():
    print(f'Data need: {data_need_name}')
    rows = country_vehicle[country_vehicle['Data need'] == data_need_name]
    assert (rows.Units == '%').all()
    assert (rows['Data point name'] == 'percentage').all()

    result = check_with_total(rows)
    result.value = result.value.clip(0, 1)
    result.insert(0, "vehicle_name", "bouillon")
    result.to_csv(f'{pregnancy_sim_data_dir}/{dir_name}/nigeria.csv', index=False)

Data need: Vehicle consumption by WRA -- any
Total: 0.981
By quintile
Quintile
Lowest     0.963
Second     0.981
Middle     0.992
Fourth     0.986
Highest    0.986
Name: Value, dtype: float64
Implied total with equal weight: 0.9815999999999999
Data need: Vehicle "fortifiability" (essentially amount industrially produced)
Total: 0.9796126401630989
By quintile
Quintile
Lowest     0.965732
Second     0.978593
Middle     0.980847
Fourth     0.988844
Highest    0.983773
Name: Value, dtype: float64
Implied total with equal weight: 0.9795577532904515


In [9]:
mean_sd_data_needs = {
    "Vehicle consumption by WRA -- amount": "vehicle_consumption"
}

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [10]:
for data_need_name, dir_suffix in mean_sd_data_needs.items():
    print(f'Data need: {data_need_name}')
    rows = country_vehicle[country_vehicle['Data need'] == data_need_name]
    assert (rows.Units == 'g/day').all()
    assert (rows['Data point name'].isin(['mean', 'standard deviation'])).all()
    assert (rows.groupby(['Data point name', 'Quintile']).size() == 1).all()

    for stat, dir_part in [('mean', 'mean'), ('standard deviation', 'sd')]:
        stat_rows = rows[rows['Data point name'] == stat]
        result = check_with_total(stat_rows)
        result.insert(0, "vehicle_name", "bouillon")
        result.to_csv(f'{pregnancy_sim_data_dir}/{dir_part}_{dir_suffix}/nigeria.csv', index=False)

Data need: Vehicle consumption by WRA -- amount
Total: 6.3
By quintile
Quintile
Lowest     8.4
Second     8.0
Middle     5.9
Fourth     4.9
Highest    4.6
Name: Value, dtype: float64
Implied total with equal weight: 6.359999999999999


OSError: Cannot save file into a non-existent directory: '../../0200_pregnancy_sim/src/vivarium_gates_lsff_by_wealth_quintile/data/raw_data/mean_vehicle_consumption'

In [ ]:
country_vehicle_fort = pd.read_excel('./Data Extraction Sheet.xlsx', sheet_name="Country-Vehicle-Fort Extraction")
country_vehicle_fort = country_vehicle_fort[(country_vehicle_fort.Country == 'Nigeria') & (country_vehicle_fort.Fortificant == 'Iron')]
assert (country_vehicle_fort.Vehicle == 'Bouillon').all()
country_vehicle_fort

In [ ]:
percent_data_needs = {
    "Vehicle fortification at baseline -- any": "baseline_full_fortification_coverage",
}

for data_need_name, dir_name in percent_data_needs.items():
    print(f'Data need: {data_need_name}')
    rows = country_vehicle_fort[country_vehicle_fort['Data need'] == data_need_name]
    assert (rows.Units == '%').all()
    assert (rows['Data point name'] == 'percentage').all()

    result = check_with_total(rows)
    result.value = result.value.clip(0, 1)
    result.insert(0, "vehicle_name", "bouillon")
    result.to_csv(f'{pregnancy_sim_data_dir}/{dir_name}/nigeria.csv', index=False)

In [ ]:
concentration_data_needs = {
    "Vehicle fortification at baseline -- amount among fortified": "baseline_iron_fortification_concentration",
}

for data_need_name, dir_name in concentration_data_needs.items():
    print(f'Data need: {data_need_name}')
    rows = country_vehicle_fort[country_vehicle_fort['Data need'] == data_need_name]
    assert (rows.Units == 'mcg/g').all()

    result = check_with_total(rows)
    result.insert(0, "vehicle_name", "bouillon")
    result.to_csv(f'{pregnancy_sim_data_dir}/{dir_name}/nigeria.csv', index=False)

In [ ]:
scenarios = pd.read_excel('./Data Extraction Sheet.xlsx', sheet_name="Scenario Definition Extraction")
scenarios = scenarios[(scenarios.Country == 'Nigeria') & (scenarios.Fortificant == 'Iron')]
assert (scenarios.Vehicle == 'Bouillon').all()
scenarios

In [ ]:
percent_data_needs = {
    'Intervention coverage % of fortifiable and unfortified': "intervention_iron_fortification_coverage",
    'Intervention effective % of newly fortified': "intervention_iron_fortification_effective_coverage",
}

for data_need_name, dir_name in percent_data_needs.items():
    print(f'Data need: {data_need_name}')
    rows = scenarios[scenarios['Data need'] == data_need_name]
    assert (rows.Units == '%').all()
    assert (rows['Data point name'] == 'percentage').all()
    assert (rows['Scenario'] == 'Intervention').all()

    result = rows[["Vehicle", "Value"]].rename(columns={"Vehicle": "vehicle_name", "Value": "value"})
    result.to_csv(f'{pregnancy_sim_data_dir}/{dir_name}/nigeria.csv', index=False)

In [ ]:
concentration_data_needs = {
    "Vehicle fortification in intervention -- amount among fortified": "intervention_iron_fortification_concentration",
}

for data_need_name, dir_name in concentration_data_needs.items():
    print(f'Data need: {data_need_name}')
    rows = country_vehicle_fort[country_vehicle_fort['Data need'] == data_need_name]
    assert (rows.Units == 'mcg/g').all()

    result = rows[["Vehicle", "Value"]].rename(columns={"Vehicle": "vehicle_name", "Value": "value"})
    result.to_csv(f'{pregnancy_sim_data_dir}/{dir_name}/nigeria.csv', index=False)